In [1]:
from itertools import permutations, product
import re
from collections import defaultdict

import numpy as np
import pandas as pd
from pyqubo import Array, Constraint, Placeholder
import neal

from data import PoCData

# データ入力

In [2]:
poc_data = PoCData()
orders, areas, vehicles = poc_data.get_data()

AttributeError: 'Series' object has no attribute 'load_read_id'

In [ ]:
orders

In [ ]:
orders_df = pd.DataFrame(orders)
orders_df

In [ ]:
vehicles_df = pd.DataFrame(vehicles)
vehicles_df

# 前処理

In [ ]:
# 日別にデータを分割する。
orders_df['start_time'] = pd.to_datetime(orders_df['start_time'])
orders_df['end_time'] = pd.to_datetime(orders_df['end_time'])
order_date_groups = orders_df.groupby(orders_df['start_time'].dt.date)
order_date_groups_df = [order_date_group for _, order_date_group in order_date_groups]

orders = {}
for i in range(len(order_date_groups_df)):
    orders[i] = poc_data.get_order(order_date_groups_df[i].reset_index(drop=True))
orders

- 距離のマトリクス (時間と同様の扱いとする)

In [ ]:
def get_dist_matrix(day_orders, day_order_ids):
    dist_matrix = {}
    for j, k in permutations(day_order_ids, 2):
        dist = np.linalg.norm(np.array(day_orders[j].coordinate) - np.array(day_orders[k].coordinate), ord=1)
        dist_matrix[(j, k)] = round(dist, 1)*10
    return dist_matrix

In [ ]:
# orderのidリストをあらかじめ用意
order_ids = {}
for i in orders:
    tmp_order_ids = []
    for order in orders[i]:
        tmp_order_ids.append(order.id)
    order_ids[i] = tmp_order_ids

In [ ]:
# オーダーに紐づく製品が同じかどうかを判定
def is_product_transportable(from_order, to_order, product_name):
    if from_order.product_name == product_name and to_order.product_name == product_name:
        return True
    return False

# 定式化・最適化

In [ ]:
# 日付ごとに最適化
root_vehicle_pair_results = {}
for d in range(len(orders)):
    
    # 日別のオーダー集合
    day_orders = orders[d]
    day_order_ids = order_ids[d]
    
    ## 1. ルートの最適化
    print("===== Start  1.Root Optimize =====")
    print(day_orders[0].start_time)

    # 距離マトリクス
    dist_matrix = get_dist_matrix(day_orders, day_order_ids)

    # 決定変数: ルート候補i においてオーダーjからオーダーkへ移動するかどうか？
    n_root_candidate = 10 # ルート候補数
    x = Array.create("x", shape=(n_root_candidate, len(possible_pairs), len(possible_pairs)), vartype='BINARY')

    # 定式化処理
    opt1_obj1 = 0
    opt1_H1 = 0
    opt1_H2 = 0
    opt1_H3 = 0
    opt1_H4 = 0
    for i in range(n_root_candidate):
        
        # 目的関数: 各ルート候補の移動時間を最小化
        opt1_obj1 += sum(dist_matrix[j, k]*x[i, j, k] for j, k in possible_pairs)
                
        # 制約条件: 各ルート候補のオーダー間の移動合計時間は指定時間内
        opt1_H1_tmp = 0
        target_work_time = 13
        for j, k in possible_pairs:
            opt1_H1_tmp += dist_matrix[j, k]*x[i, j, k] 
        opt1_H1 += Constraint((opt1_H1_tmp - target_work_time)**2, f"opt1_H1")
        
        # 制約条件: 同じ製品を輸送するオーダー間で移動すること
        products = ['product_0', 'product_1', 'product_2']
        for product_name in products:
            for j, k in possible_pairs:
                opt1_H2 += Constraint((is_product_transportable(day_orders[j], day_orders[k], product_name)*x[i, j, k] - 1)**2, f"opt1_H2")
        
        # 制約条件: オーダーに紐づく製品の重さに制限がある
        product_weight_limit = 30000
        for j, k in possible_pairs:
            opt1_H3 += Constraint((is_product_weight(day_orders[j], day_orders[k], product_weight_limit)*x[i, j, k] - 1)**2, f"opt1_H3")
        
        # 制約条件: 連続性の保証
        for from_order_pair, to_order_pair in product(possible_pairs, repeat=2):
            
            if from_order_pair == to_order_pair:
                continue
            
            j1 = from_order_pair[0]
            k1 = from_order_pair[1]
            j2 = to_order_pair[0]
            k2 = to_order_pair[1]

            # 以下の場合は、つなげないようにする。
            if j1 == k1 or j2 == k2 or j1 == k2:
                opt1_H4 += Constraint(x[i, j1, k1]*x[i, j2, k2], f"opt1_H4_1")

            # 同じ末尾と先頭のオーダー同士で連続的につなぐことを保証させる。
            if k1 == j2:
                opt1_H4 += Constraint((x[i, j1, k1]*x[i, j2, k2] - 1)**2, f"opt1_H4_2")
            
    H = opt1_obj1 + opt1_H1 + opt1_H2 + opt1_H3 + opt1_H4

    # QUBO生成
    model = H.compile()
    qubo, offset = model.to_qubo()

    # アニーリング
    sampler = neal.SimulatedAnnealingSampler()
    result = sampler.sample_qubo(qubo)
    result = result.first.sample
    
    # 制約チェック
    decoded_sample = model.decode_sample(result, vartype="BINARY")
    print("constraints check :", decoded_sample.constraints(only_broken=True))
    
    
    # 結果の加工
    active_spin_keys = []
    roots = defaultdict(list)
    for key in result:
        if result[key] == 1:
            active_spin_keys.append(key)
            
    for key in active_spin_keys:
        indices = re.findall(r'\d+', key)
        indices = list(map(int, indices))
        roots[indices[0]].append((indices[1:]))
    roots = dict(roots)
    
    for i in range(len(roots)):
        dist = 0.0
        for root in roots[i]:
            dist += dist_matrix[root[0], root[1]]
        print("root candidate", i, ": ", roots[i], "total distance: ", dist)
        
    print("===== End  1.Root Optimize =====\n")
    

    ## 2. ルートと車両の割り当て
    print("===== Start  2.Root and Vehicle Optimize =====")
    
    # # 決定変数: ルートiに車両yを割り当てるかどうか？
    y = Array.create("y", shape=(len(roots), len(vehicles)), vartype='BINARY')

    # # 目的関数: 車両数はなるべく少なくしたい
    opt2_obj1 = sum(y[i, j] for i in range(len(roots)) for j in range(len(vehicles)))

    # # 制約条件: ルートと車両は1:1
    opt2_H1 = 0
    for i in range(len(roots)):
        opt2_H1 += Constraint((sum(y[i, j] for j in range(len(vehicles))) - 1)**2, f"opt2_H1_1")
    for j in range(len(vehicles)):
        opt2_H1 += Constraint((sum(y[i, j] for i in range(len(roots))) - 1)**2, f"opt2_H1_2")
        
    H = opt2_obj1 + opt2_H1

    # # QUBO生成
    model = H.compile()
    qubo, offset = model.to_qubo()

    # # アニーリング
    sampler = neal.SimulatedAnnealingSampler()
    result = sampler.sample_qubo(qubo)
    result = result.first.sample

    # 制約チェック
    decoded_sample = model.decode_sample(result, vartype="BINARY")
    print("constraints check :", decoded_sample.constraints(only_broken=True))
    
    # 結果の加工
    vehicle_root_pairs = {}
    for i in range(len(roots)):
        for j in range(len(vehicles)):
            key = f"y[{i}][{j}]"
            matches = re.findall(r'\d+', key)
            indices = list(map(int, matches))
            if result[key] == 1:
                print(vehicles[indices[1]].vehicle_name, roots[indices[0]])
                vehicle_root_pairs[vehicles[indices[1]].id] = roots[indices[0]]
                
    root_vehicle_pair_results[d] = vehicle_root_pairs
    print("===== End  2.Root and Vehicle Optimize =====\n")

In [ ]:
# 日別の車両とルートの最適化結果
root_vehicle_pair_results

In [ ]:
# 全計画作成
results = {}
for i in range(len(root_vehicle_pair_results)):
    print(root_vehicle_pair_results[i])

## 可視化